# Agricultural Data Generator for Cameroon

Generate millions of clean, coherent agricultural records based on Cameroonian crop taxonomy.

## Features:
- 100+ different crops across all categories
- Agroecological zone consistency
- Climate-adapted varieties
- Realistic yield correlations
- Clean, validated output

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta, date
import json
import os
import sys
from typing import Dict, List, Any, Tuple
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

sys.path.append('../')
CONFIG_DIR = '../config/json/'
OUTPUT_DIR = '../data/generated/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def load_schemas():
    schemas = {}
    
    with open(f'{CONFIG_DIR}agricultural_taxonomy.json', 'r', encoding='utf-8') as f:
        content = f.read().replace("agricultural_taxonomy = ", "").replace("True", "true").replace("False", "false").replace("'", '"')
        schemas['taxonomy'] = json.loads(content)
    
    with open(f'{CONFIG_DIR}weather_data_schema.json', 'r', encoding='utf-8') as f:
        content = f.read().replace("weather_data_schema = ", "").replace("'", '"')
        schemas['weather'] = json.loads(content)
    
    with open(f'{CONFIG_DIR}soil_data_schema.json', 'r', encoding='utf-8') as f:
        content = f.read().replace("soil_data_schema = ", "").replace("'", '"')
        schemas['soil'] = json.loads(content)
    
    with open(f'{CONFIG_DIR} remote_sensing_schema.json', 'r', encoding='utf-8') as f:
        content = f.read().replace("remote_sensing_schema = ", "").replace("True", "true").replace("False", "false").replace("'", '"')
        schemas['remote_sensing'] = json.loads(content)
    
    return schemas

schemas = load_schemas()

In [ ]:
CAMEROON_COORDS = {
    'lat_range': (1.6, 13.1),
    'lon_range': (8.5, 16.2),
    'elevation_ranges': {
        'coastal': (0, 200),
        'plateau': (200, 800),
        'highland': (800, 2500),
        'mountain': (2500, 4095)
    }
}

AGROECOLOGICAL_ZONES = {
    'coastal_forest': {
        'lat_range': (1.6, 4.5),
        'rainfall_mm': (1500, 4000),
        'temp_range': (22, 32),
        'suitable_crops': ['cocoa', 'oil_palm', 'plantain_banana', 'cassava', 'yam', 'sweet_potato', 'papaya', 'pineapple']
    },
    'forest_savanna': {
        'lat_range': (4.5, 7.0),
        'rainfall_mm': (1200, 1800),
        'temp_range': (20, 30),
        'suitable_crops': ['maize', 'groundnut', 'yam', 'tomato', 'pepper', 'okra', 'cucumber', 'common_bean', 'soybean']
    },
    'guinea_savanna': {
        'lat_range': (7.0, 10.0),
        'rainfall_mm': (800, 1400),
        'temp_range': (18, 35),
        'suitable_crops': ['sorghum', 'millet', 'cowpea', 'cotton', 'onion', 'sesame', 'sunflower']
    },
    'sahel_savanna': {
        'lat_range': (10.0, 13.1),
        'rainfall_mm': (200, 800),
        'temp_range': (15, 40),
        'suitable_crops': ['millet', 'sorghum', 'cowpea', 'cotton', 'dates', 'moringa']
    },
    'highlands': {
        'lat_range': (5.0, 7.5),
        'elevation': (800, 2500),
        'rainfall_mm': (1000, 2500),
        'temp_range': (10, 25),
        'suitable_crops': ['potato', 'coffee', 'cabbage', 'carrot', 'lettuce', 'beans', 'peas', 'wheat']
    },
    'mont_cameroun_volcanic': {
        'lat_range': (4.0, 4.35),
        'lon_range': (9.0, 9.35),
        'elevation': (2500, 4095),
        'rainfall_mm': (2000, 5000),
        'temp_range': (5, 18),
        'suitable_crops': ['potato', 'cabbage', 'carrot', 'lettuce']
    }
}

In [ ]:
class DataGenerator:
    def __init__(self, schemas: Dict):
        self.schemas = schemas
        self.field_counter = 0
        self.all_crops = self._extract_all_crops()
    
    def _extract_all_crops(self) -> List[str]:
        crops = []
        taxonomy = self.schemas['taxonomy']
        
        for category in taxonomy.values():
            if 'crops' in category:
                crops.extend(category['crops'].keys())
        
        return crops
    
    def generate_field_id(self, prefix: str = "CAM") -> str:
        self.field_counter += 1
        return f"{prefix}_{self.field_counter:08d}"
    
    def select_zone_and_crop(self) -> Tuple[str, Dict, str]:
        zone_name = random.choice(list(AGROECOLOGICAL_ZONES.keys()))
        zone_info = AGROECOLOGICAL_ZONES[zone_name]
        
        suitable_crops = zone_info.get('suitable_crops', self.all_crops)
        available_crops = [c for c in suitable_crops if c in self.all_crops]
        
        if not available_crops:
            available_crops = random.sample(self.all_crops, min(10, len(self.all_crops)))
        
        crop_type = random.choice(available_crops)
        return zone_name, zone_info, crop_type
    
    def generate_coordinates(self, zone_info: Dict) -> Tuple[float, float, float]:
        lat_min, lat_max = zone_info['lat_range']
        latitude = round(random.uniform(lat_min, lat_max), 6)
        
        # Use zone-specific lon_range if available, otherwise default
        if 'lon_range' in zone_info:
            longitude = round(random.uniform(*zone_info['lon_range']), 6)
        else:
            longitude = round(random.uniform(*CAMEROON_COORDS['lon_range']), 6)
        
        if 'elevation' in zone_info:
            elevation = random.uniform(*zone_info['elevation'])
        elif latitude < 3:
            elevation = random.uniform(*CAMEROON_COORDS['elevation_ranges']['coastal'])
        elif latitude < 8:
            elevation = random.uniform(*CAMEROON_COORDS['elevation_ranges']['plateau'])
        else:
            elevation = random.uniform(*CAMEROON_COORDS['elevation_ranges']['highland'])
        
        return latitude, longitude, round(elevation, 1)
    
    def get_crop_info(self, crop_type: str) -> Dict:
        taxonomy = self.schemas['taxonomy']
        
        for category in taxonomy.values():
            if 'crops' in category and crop_type in category['crops']:
                return category['crops'][crop_type]
        
        return {}
    
    def generate_dates_and_cycle(self, crop_type: str, year: int) -> Tuple[date, date, int]:
        crop_info = self.get_crop_info(crop_type)
        
        if 'planting_seasons' in crop_info:
            main_season = crop_info['planting_seasons']['main']
            if 'March-June' in main_season:
                planting_month = random.choice([3, 4, 5, 6])
            elif 'August-September' in main_season:
                planting_month = random.choice([8, 9])
            else:
                planting_month = random.choice([3, 4, 5, 8, 9])
        else:
            planting_month = random.choice([3, 4, 5, 8, 9])
        
        planting_date = date(year, planting_month, random.randint(1, 28))
        
        if 'cycle_days' in crop_info:
            cycle_length = random.randint(crop_info['cycle_days']['min'], crop_info['cycle_days']['max'])
        else:
            cycle_length = random.randint(60, 150)
        
        harvest_date = planting_date + timedelta(days=cycle_length)
        return planting_date, harvest_date, cycle_length
    
    def calculate_yield(self, crop_type: str, inputs: Dict, zone_info: Dict) -> Tuple[float, float, float]:
        crop_info = self.get_crop_info(crop_type)
        
        if 'yield_potential' in crop_info:
            base_min = crop_info['yield_potential']['min']
            base_max = crop_info['yield_potential']['max']
            base_avg = crop_info['yield_potential']['average']
        else:
            base_min, base_max, base_avg = 0.5, 8.0, 3.0
        
        fertilizer_effect = 1.0 + min(inputs.get('nitrogen_kg_ha', 0) / 150, 0.6)
        fertilizer_effect += min(inputs.get('phosphorus_kg_ha', 0) / 60, 0.4)
        fertilizer_effect += min(inputs.get('organic_fertilizer_tha', 0) / 12, 0.5)
        
        irrigation_effect = 1.3 if inputs.get('irrigation', False) else 1.0
        zone_effect = random.uniform(0.7, 1.3)
        random_factor = random.uniform(0.5, 1.5)
        
        final_yield = base_avg * fertilizer_effect * irrigation_effect * zone_effect * random_factor
        final_yield = max(base_min, min(base_max * 1.5, final_yield))
        
        harvest_index = random.uniform(0.2, 0.6)
        biomass = final_yield / harvest_index
        
        return round(final_yield, 2), round(biomass, 2), round(harvest_index, 3)

In [ ]:
from utils.date_utils import get_agricultural_season

def generate_crop_data(generator: DataGenerator, num_records: int) -> pd.DataFrame:
    records = []
    
    for i in range(num_records):
        zone_name, zone_info, crop_type = generator.select_zone_and_crop()
        latitude, longitude, elevation = generator.generate_coordinates(zone_info)
        
        field_id = generator.generate_field_id()
        year = random.randint(2018, 2024)
        
        crop_info = generator.get_crop_info(crop_type)
        varieties = crop_info.get('varieties_cameroon', [f'Local_{crop_type}', f'Improved_{crop_type}'])
        variety = random.choice(varieties)
        
        planting_date, harvest_date, cycle_length = generator.generate_dates_and_cycle(crop_type, year)
        
        area_hectares = round(random.uniform(0.1, 20), 2)
        
        # Use get_agricultural_season instead of random season
        season = get_agricultural_season(planting_date, latitude)
        
        irrigation = random.choice([True, False])
        nitrogen_kg_ha = round(random.uniform(0, 250 if irrigation else 120), 1)
        phosphorus_kg_ha = round(random.uniform(0, 100 if irrigation else 50), 1)
        potassium_kg_ha = round(random.uniform(0, 80 if irrigation else 40), 1)
        organic_fertilizer_tha = round(random.uniform(0, 20 if irrigation else 10), 1)
        
        inputs = {
            'nitrogen_kg_ha': nitrogen_kg_ha,
            'phosphorus_kg_ha': phosphorus_kg_ha,
            'organic_fertilizer_tha': organic_fertilizer_tha,
            'irrigation': irrigation
        }
        
        yield_tha, biomass_tha, harvest_index = generator.calculate_yield(crop_type, inputs, zone_info)
        
        cultivation_system = random.choice(['open_field', 'greenhouse', 'shade_house']) if crop_type in ['tomato', 'pepper', 'cucumber'] else 'open_field'
        pest_pressure = random.choice(['low', 'moderate', 'high'])
        
        # Humidity-correlated disease incidence
        # Estimate humidity from zone: coastal/forest zones are more humid
        if zone_name in ('coastal_forest', 'mont_cameroun_volcanic'):
            est_humidity = random.uniform(75, 95)
        elif zone_name in ('forest_savanna', 'highlands'):
            est_humidity = random.uniform(60, 85)
        elif zone_name == 'guinea_savanna':
            est_humidity = random.uniform(45, 75)
        else:  # sahel
            est_humidity = random.uniform(25, 60)
        
        if est_humidity > 80:
            disease_incidence = round(random.uniform(10, 40), 1)
        elif est_humidity > 60:
            disease_incidence = round(random.uniform(5, 25), 1)
        else:
            disease_incidence = round(random.uniform(0, 15), 1)
        
        record = {
            'field_id': field_id,
            'crop_type': crop_type,
            'variety': variety,
            'year': year,
            'season': season,
            'latitude': latitude,
            'longitude': longitude,
            'elevation': elevation,
            'agroecological_zone': zone_name,
            'area_hectares': area_hectares,
            'planting_date': planting_date,
            'harvest_date': harvest_date,
            'cycle_length': cycle_length,
            'cultivation_system': cultivation_system,
            'nitrogen_kg_ha': nitrogen_kg_ha,
            'phosphorus_kg_ha': phosphorus_kg_ha,
            'potassium_kg_ha': potassium_kg_ha,
            'organic_fertilizer_tha': organic_fertilizer_tha,
            'irrigation': irrigation,
            'pest_pressure': pest_pressure,
            'disease_incidence': disease_incidence,
            'yield_tha': yield_tha,
            'biomass_tha': biomass_tha,
            'harvest_index': harvest_index
        }
        
        records.append(record)
    
    return pd.DataFrame(records)

In [ ]:
def generate_weather_data(generator: DataGenerator, num_records: int) -> pd.DataFrame:
    records = []
    start_date = datetime(2018, 1, 1)
    
    stations = []
    for i in range(50):
        zone_name = random.choice(list(AGROECOLOGICAL_ZONES.keys()))
        zone_info = AGROECOLOGICAL_ZONES[zone_name]
        lat, lon, elev = generator.generate_coordinates(zone_info)
        stations.append({
            'station_id': f'CAM_METEO_{i+1:03d}',
            'latitude': lat,
            'longitude': lon,
            'elevation': elev,
            'zone': zone_name,
            'zone_info': zone_info
        })
    
    # Pre-generate daily spatial offsets for correlated weather within same zone
    num_days = 2501
    zone_daily_offsets = {}
    for zone_name in AGROECOLOGICAL_ZONES:
        zone_daily_offsets[zone_name] = {
            'temp': np.random.normal(0, 1.5, num_days),
            'precip': np.random.normal(0, 0.15, num_days),
        }
    
    for i in range(num_records):
        station = random.choice(stations)
        day_idx = random.randint(0, 2500)
        current_date = start_date + timedelta(days=day_idx)
        
        month = current_date.month
        is_dry_season = month in [11, 12, 1, 2, 3]
        is_wet_season = month in [6, 7, 8, 9]
        
        zone_info = station['zone_info']
        base_temp_min, base_temp_max = zone_info['temp_range']
        
        # Temperature-altitude correlation: -6.5°C per 1000m
        temp_adjustment = -6.5 * (station['elevation'] / 1000)
        seasonal_adjustment = random.uniform(-2, 5) if is_dry_season else random.uniform(-3, 2)
        
        # Add zone-level spatial correlation offset
        zone_temp_offset = zone_daily_offsets[station['zone']]['temp'][day_idx]
        
        temp_min = round(base_temp_min + temp_adjustment + seasonal_adjustment + zone_temp_offset + random.uniform(-1.5, 1.5), 1)
        temp_max = round(base_temp_max + temp_adjustment + seasonal_adjustment + zone_temp_offset + random.uniform(-1, 2), 1)
        
        if temp_max <= temp_min:
            temp_max = temp_min + random.uniform(3, 8)
        
        temp_avg = round((temp_min + temp_max) / 2, 1)
        
        if is_dry_season:
            precip_prob, max_precip = 0.1, 20
        elif is_wet_season:
            precip_prob, max_precip = 0.7, 150
        else:
            precip_prob, max_precip = 0.4, 80
        
        # Spatial correlation for precipitation
        zone_precip_offset = zone_daily_offsets[station['zone']]['precip'][day_idx]
        adjusted_precip_prob = max(0, min(1, precip_prob + zone_precip_offset))
        
        if random.random() < adjusted_precip_prob:
            precipitation_daily = round(random.uniform(0.1, max_precip), 1)
            precipitation_intensity = round(precipitation_daily / random.uniform(1, 6), 1)
        else:
            precipitation_daily = 0.0
            precipitation_intensity = 0.0
        
        base_humidity = 85 if station['zone'] in ('coastal_forest', 'mont_cameroun_volcanic') else 60
        humidity_adj = random.uniform(-20, -5) if is_dry_season else random.uniform(-5, 10)
        relative_humidity = max(20, min(100, base_humidity + humidity_adj))
        
        base_radiation = 25 if is_dry_season else 18
        cloud_factor = 1 - (precipitation_daily / 100) * 0.7
        solar_radiation = round(base_radiation * cloud_factor * random.uniform(0.8, 1.2), 1)
        
        wind_speed = round(random.uniform(0.5, 15), 1)
        atmospheric_pressure = round(1013 - (station['elevation'] * 0.12) + random.uniform(-15, 15), 1)
        
        # Use canonical season function
        season = get_agricultural_season(current_date.date(), station['latitude'])
        
        record = {
            'station_id': station['station_id'],
            'date': current_date.date(),
            'latitude': station['latitude'],
            'longitude': station['longitude'],
            'elevation': station['elevation'],
            'agroecological_zone': station['zone'],
            'season': season,
            'temperature_min': temp_min,
            'temperature_max': temp_max,
            'temperature_avg': temp_avg,
            'precipitation_daily': precipitation_daily,
            'precipitation_intensity': precipitation_intensity,
            'relative_humidity': round(relative_humidity, 1),
            'solar_radiation': solar_radiation,
            'wind_speed': wind_speed,
            'atmospheric_pressure': atmospheric_pressure
        }
        
        records.append(record)
    
    return pd.DataFrame(records)

In [ ]:
def generate_soil_data(generator: DataGenerator, num_records: int) -> pd.DataFrame:
    records = []
    
    for i in range(num_records):
        zone_name = random.choice(list(AGROECOLOGICAL_ZONES.keys()))
        zone_info = AGROECOLOGICAL_ZONES[zone_name]
        latitude, longitude, elevation = generator.generate_coordinates(zone_info)
        
        sample_id = f"SOIL_{generator.generate_field_id().split('_')[1]}"
        year = random.randint(2018, 2024)
        depth_cm = random.choice([10, 20, 30])
        
        if zone_name == 'mont_cameroun_volcanic':
            # Andosols: volcanic soils with high organic matter
            clay_base = random.uniform(15, 35)
            sand_base = random.uniform(30, 55)
        elif zone_name == 'coastal_forest':
            clay_base = random.uniform(25, 60)
            sand_base = random.uniform(20, 80)
        elif zone_name == 'sahel_savanna':
            clay_base = random.uniform(5, 25)
            sand_base = random.uniform(20, 80)
        else:
            clay_base = random.uniform(15, 40)
            sand_base = random.uniform(20, 80)
        
        silt_base = 100 - clay_base - sand_base
        
        if silt_base < 0:
            sand_base += silt_base
            silt_base = 0
        
        total = sand_base + silt_base + clay_base
        sand_percentage = round(sand_base / total * 100, 1)
        silt_percentage = round(silt_base / total * 100, 1)
        clay_percentage = round(100 - sand_percentage - silt_percentage, 1)
        
        if clay_percentage >= 40:
            texture_class = 'clay'
        elif clay_percentage >= 27:
            texture_class = 'clay_loam'
        elif clay_percentage >= 7:
            texture_class = 'loam'
        else:
            texture_class = 'sandy_loam'
        
        if zone_name == 'mont_cameroun_volcanic':
            # Andosols: slightly acidic, high organic matter
            ph_water = round(random.uniform(5.5, 6.5), 1)
            organic_carbon = round(random.uniform(3.0, 8.0), 2)
        elif zone_name == 'coastal_forest':
            ph_water = round(random.uniform(4.5, 6.0), 1)
            organic_carbon = round(random.uniform(2.0, 6.0), 2)
        elif zone_name == 'sahel_savanna':
            ph_water = round(random.uniform(6.0, 8.0), 1)
            organic_carbon = round(random.uniform(0.3, 2.0), 2)
        else:
            ph_water = round(random.uniform(5.0, 7.0), 1)
            organic_carbon = round(random.uniform(0.8, 4.0), 2)
        
        organic_matter = round(organic_carbon * 1.724, 2)
        total_nitrogen = round(organic_carbon / random.uniform(10, 15), 3)
        
        available_phosphorus = round(random.uniform(2, 30), 1)
        exchangeable_potassium = round(random.uniform(0.05, 1.5), 2)
        cation_exchange_capacity = round(clay_percentage * 0.5 + organic_carbon * 2, 1)
        
        record = {
            'sample_id': sample_id,
            'latitude': latitude,
            'longitude': longitude,
            'elevation': elevation,
            'agroecological_zone': zone_name,
            'year': year,
            'depth_cm': depth_cm,
            'sand_percentage': sand_percentage,
            'silt_percentage': silt_percentage,
            'clay_percentage': clay_percentage,
            'texture_class': texture_class,
            'ph_water': ph_water,
            'organic_carbon': organic_carbon,
            'organic_matter': organic_matter,
            'total_nitrogen': total_nitrogen,
            'available_phosphorus': available_phosphorus,
            'exchangeable_potassium': exchangeable_potassium,
            'cation_exchange_capacity': cation_exchange_capacity
        }
        
        records.append(record)
    
    return pd.DataFrame(records)

In [ ]:
def clean_data(df: pd.DataFrame, dataset_type: str) -> pd.DataFrame:
    initial_count = len(df)
    
    if 'field_id' in df.columns:
        df = df.drop_duplicates(subset=['field_id'])
    elif 'sample_id' in df.columns:
        df = df.drop_duplicates(subset=['sample_id'])
    elif 'station_id' in df.columns and 'date' in df.columns:
        df = df.drop_duplicates(subset=['station_id', 'date'])
    
    if dataset_type == 'crops':
        df = df[df['harvest_date'] > df['planting_date']]
        df = df[df['yield_tha'] <= df['biomass_tha']]
        df = df[df['area_hectares'] > 0]
        df = df[df['harvest_index'].between(0.1, 0.8)]
        df = df[df['yield_tha'].between(0.1, 50)]
        
    elif dataset_type == 'weather':
        df = df[df['temperature_max'] > df['temperature_min']]
        df = df[df['relative_humidity'].between(0, 100)]
        df = df[df['precipitation_daily'] >= 0]
        df = df[df['solar_radiation'] >= 0]
        
    elif dataset_type == 'soil':
        texture_sum = df['sand_percentage'] + df['silt_percentage'] + df['clay_percentage']
        df = df[abs(texture_sum - 100) < 2]
        df = df[df['ph_water'].between(3.5, 9.5)]
        df = df[df['organic_carbon'] >= 0.1]
    
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df = df[df['latitude'].between(*CAMEROON_COORDS['lat_range'])]
        df = df[df['longitude'].between(*CAMEROON_COORDS['lon_range'])]
    
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    for col in numeric_columns:
        if df[col].dtype in ['float64', 'float32']:
            if col in ['latitude', 'longitude']:
                df[col] = df[col].round(6)
            elif col in ['yield_tha', 'biomass_tha', 'temperature_min', 'temperature_max']:
                df[col] = df[col].round(2)
            else:
                df[col] = df[col].round(1)
    
    return df

In [ ]:
GENERATION_CONFIG = {
    'crops': {'records': 2_000_000, 'filename': 'crop_data_cameroon.csv'},
    'weather': {'records': 1_500_000, 'filename': 'weather_data_cameroon.csv'},
    'soil': {'records': 500_000, 'filename': 'soil_data_cameroon.csv'}
}

generator = DataGenerator(schemas)
start_time = datetime.now()

for dataset_type, config in GENERATION_CONFIG.items():
    if dataset_type == 'crops':
        df = generate_crop_data(generator, config['records'])
    elif dataset_type == 'weather':
        df = generate_weather_data(generator, config['records'])
    elif dataset_type == 'soil':
        df = generate_soil_data(generator, config['records'])
    
    df_clean = clean_data(df, dataset_type)
    
    filename = os.path.join(OUTPUT_DIR, config['filename'])
    df_clean.to_csv(filename, index=False, encoding='utf-8')
    
    file_size_mb = os.path.getsize(filename) / (1024*1024)
    print(f"{dataset_type.upper()}: {len(df_clean):,} records, {file_size_mb:.1f} MB")
    
    del df, df_clean

total_duration = datetime.now() - start_time
print(f"\nGeneration completed in {total_duration}")

In [ ]:
# Validation
for dataset_type, config in GENERATION_CONFIG.items():
    filename = os.path.join(OUTPUT_DIR, config['filename'])
    if os.path.exists(filename):
        sample_df = pd.read_csv(filename, nrows=1000)
        
        print(f"\n{dataset_type.upper()}:")
        print(f"  Columns: {len(sample_df.columns)}")
        print(f"  Missing values: {sample_df.isnull().sum().sum()}")
        
        if dataset_type == 'crops':
            print(f"  Unique crops: {sample_df['crop_type'].nunique()}")
            print(f"  Sample crops: {list(sample_df['crop_type'].unique()[:10])}")
            print(f"  Avg yield: {sample_df['yield_tha'].mean():.2f} t/ha")
        
        elif dataset_type == 'weather':
            print(f"  Stations: {sample_df['station_id'].nunique()}")
            print(f"  Avg temp: {sample_df['temperature_avg'].mean():.1f}°C")
        
        elif dataset_type == 'soil':
            print(f"  Avg pH: {sample_df['ph_water'].mean():.1f}")
            print(f"  Texture classes: {sample_df['texture_class'].nunique()}")

print(f"\nFiles generated in: {OUTPUT_DIR}")
print("Ready for analysis and ML training!")